# NTFS Timestomping Detection Tool v2.0
# Step 2: Feature Engineering

---

## Overview

This notebook extracts 51+ forensic features from the merged data:
- **Timestamp Analysis**: Deltas, direction changes, zero nanoseconds
- **File Characteristics**: Extensions, attributes, path depth
- **Event Patterns**: Frequency, temporal clustering
- **Cross-Artifact Validation**: LogFile + UsnJrnl consistency scores

These features enable the ML model to detect timestamp manipulation patterns.

---

## Input

`merged_forensic_data.csv` from Step 1

## Output

`forensic_features.csv` - Ready for ML detection

---

## 1. Setup

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from datetime import timedelta
import re

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


## 2. Load Merged Data

In [23]:
# Input/Output paths
INPUT_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/Lone-Wolf')
OUTPUT_DIR = INPUT_DIR  # Same directory

input_file = INPUT_DIR / 'merged_forensic_data.csv'

print("=" * 80)
print("LOADING MERGED DATA")
print("=" * 80)
print(f"\nInput: {input_file}")

if not input_file.exists():
    raise FileNotFoundError(f"Input file not found: {input_file}\nPlease run 01_Load_Data.ipynb first!")

df = pd.read_csv(input_file, low_memory=False)

print(f"\n✓ Loaded {len(df):,} records")
print(f"✓ {len(df.columns)} columns")
print(f"\nSource distribution:")
print(df['source'].value_counts())

LOADING MERGED DATA

Input: /Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/Lone-Wolf/merged_forensic_data.csv

✓ Loaded 33,888 records
✓ 25 columns

Source distribution:
source
usnjrnl_only    33301
both              433
logfile_only      154
Name: count, dtype: int64


## 3. Feature Engineering Functions

These functions extract forensic features used by the trained model.

In [24]:
def parse_lf_detail_field(df):
    """
    Parse LogFile lf_detail field to extract timestamp changes.
    
    CRITICAL: Must match Phase 2A training dtypes EXACTLY.
    Uses VECTORIZED operations to maintain correct dtypes.
    Preserves full timestamp precision.
    """
    print("\n  Parsing lf_detail field...")
    
    # Initialize timestamp columns as OBJECT dtype
    for ts_type in ['creation', 'modified', 'accessed', 'mft_modified']:
        df[f'lf_{ts_type}_time_before'] = pd.Series([None] * len(df), dtype='object')
        df[f'lf_{ts_type}_time_after'] = pd.Series([None] * len(df), dtype='object')
    
    # Use VECTORIZED operations to create bool dtype columns
    df['zero_in_nanoseconds'] = df['lf_detail'].fillna('').str.contains(
        'Zero in 100-nanoseconds', case=False, na=False
    )
    df['copied_from_file'] = df['lf_detail'].fillna('').str.contains(
        'Copied from File|same as', case=False, na=False
    )
    
    # Regex pattern for timestamp manipulation
    pattern = r'(\w+Time)\s*:\s*([\d\-:\s]+)\s*->\s*([\d\-:\s]+)'
    
    parsed_count = 0
    for idx, row in df[df['lf_detail'].notna()].iterrows():
        detail = str(row['lf_detail'])
        
        # Use findall() to capture ALL timestamp types
        matches = re.findall(pattern, detail)
        
        for match in matches:
            ts_type = match[0].replace('Time', '').lower()
            if ts_type == 'mftmodified':
                ts_type = 'mft_modified'
            
            # Store with full precision (no truncation)
            df.at[idx, f'lf_{ts_type}_time_before'] = match[1].strip()
            df.at[idx, f'lf_{ts_type}_time_after'] = match[2].strip()
            parsed_count += 1
    
    # Force object dtype
    for ts_type in ['creation', 'modified', 'accessed', 'mft_modified']:
        df[f'lf_{ts_type}_time_before'] = df[f'lf_{ts_type}_time_before'].astype('object')
        df[f'lf_{ts_type}_time_after'] = df[f'lf_{ts_type}_time_after'].astype('object')
    
    print(f"    ✓ Parsed {parsed_count:,} timestamp entries")
    print(f"    ✓ Zero nanoseconds: {df['zero_in_nanoseconds'].sum():,} events (dtype: {df['zero_in_nanoseconds'].dtype})")
    print(f"    ✓ Copied timestamps: {df['copied_from_file'].sum():,} events (dtype: {df['copied_from_file'].dtype})")
    
    # Show breakdown
    for ts_type in ['creation', 'modified', 'accessed', 'mft_modified']:
        count = df[f'lf_{ts_type}_time_before'].notna().sum()
        if count > 0:
            unique_count = df[f'lf_{ts_type}_time_before'].nunique()
            print(f"    ✓ {ts_type}: {count:,} events ({unique_count} unique values)")
    
    return df

In [25]:
def extract_timestamp_delta_features(df):
    """
    Calculate timestamp change magnitudes and directions.
    
    CRITICAL: Must match Phase 2A training dtypes EXACTLY.
    Training has BOOL dtype for *_changed_to_past columns.
    """
    print("\n  Extracting timestamp delta features...")
    
    for ts_type in ['creation', 'modified', 'accessed', 'mft_modified']:
        before_col = f'lf_{ts_type}_time_before'
        after_col = f'lf_{ts_type}_time_after'
        delta_col = f'{ts_type}_time_delta_days'
        direction_col = f'{ts_type}_time_changed_to_past'
        
        # Parse timestamps
        before_dt = pd.to_datetime(df[before_col], errors='coerce')
        after_dt = pd.to_datetime(df[after_col], errors='coerce')
        
        # Calculate delta in days (float64 with NaN)
        delta = (before_dt - after_dt).dt.total_seconds() / 86400
        df[delta_col] = delta
        
        # CRITICAL FIX: Training data has BOOL dtype (not float64 or object!)
        # Use vectorized boolean comparison - results in bool dtype
        df[direction_col] = delta > 0  # This creates bool dtype with NaN → False conversion
        
    
    # Count non-null deltas
    delta_cols = [f'{t}_time_delta_days' for t in ['creation', 'modified', 'accessed', 'mft_modified']]
    total_deltas = df[delta_cols].notna().sum().sum()
    print(f"    ✓ Calculated {total_deltas:,} timestamp deltas")
    
    return df


In [26]:
def extract_file_characteristics(df):
    """
    Extract file-based features: extension, attributes, path depth.
    
    CRITICAL: Training separates "suspicious" from "executable"!
    - Suspicious = temporary/backup files indicating cleanup (.tmp, .log, .bak, etc.)
    - Executable = code files (.exe, .dll, .sys, etc.)
    Phase 2A training does NOT include .dll as suspicious!
    """
    print("\n  Extracting file characteristics...")
    
    # CRITICAL FIX: Match Phase 2A training suspicious extensions
    # These are TEMPORARY/BACKUP files that may indicate cleanup
    # Training uses: .tmp, .log, .bak, .old, .$$$, .temp, .cache
    suspicious_extensions = ['.tmp', '.log', '.bak', '.old', '.$$$', '.temp', '.cache']
    df['has_suspicious_extension'] = df['filename'].fillna('').str.lower().apply(
        lambda x: any(x.endswith(ext) for ext in suspicious_extensions)
    )
    
    # Filename length
    df['filename_length'] = df['filename'].fillna('').str.len()
    
    # CRITICAL FIX: Check file extension as fallback if usn_file_attribute is empty
    # Match Phase 2A training executable extensions
    executable_extensions = ['.exe', '.dll', '.sys', '.com', '.scr', '.msi', '.bat', '.cmd', '.ps1', '.vbs']
    
    # Try to get from USN attribute first
    df['is_executable'] = df['usn_file_attribute'].fillna('').str.contains('Executable', case=False, na=False)
    
    # Fallback: If usn_file_attribute is empty/missing, check file extension
    empty_mask = df['usn_file_attribute'].fillna('').str.strip() == ''
    df.loc[empty_mask, 'is_executable'] = df.loc[empty_mask, 'filename'].fillna('').str.lower().apply(
        lambda x: any(x.endswith(ext) for ext in executable_extensions)
    )
    
    # File attributes from UsnJrnl (if available)
    df['is_system_file'] = df['usn_file_attribute'].fillna('').str.contains('System', case=False, na=False)
    df['is_hidden_file'] = df['usn_file_attribute'].fillna('').str.contains('Hidden', case=False, na=False)
    df['is_archive'] = df['usn_file_attribute'].fillna('').str.contains('Archive', case=False, na=False)
    
    # Path depth (number of subdirectories)
    df['path_depth'] = df['filepath'].fillna('').str.count(r'[\\|/]')
    
    print(f"    ✓ Suspicious extensions: {df['has_suspicious_extension'].sum():,} files")
    print(f"    ✓ Executable files: {df['is_executable'].sum():,}")
    print(f"    ✓ Average path depth: {df['path_depth'].mean():.1f}")
    
    return df


In [27]:
def extract_temporal_features(df):
    """
    Extract event frequency and temporal clustering features.
    CRITICAL: Must calculate windows PER-CASE to match training!
    
    Includes BOTH events_in_1min_window and events_in_5min_window.
    """
    print("\n  Extracting temporal features...")
    
    # Sort by time for windowing
    df = df.sort_values('eventtime').reset_index(drop=True)
    
    # Event frequency per file - CRITICAL: This counts how many events affect EACH unique file
    # For example: If "document.docx" appears 5 times in the dataset, this will be 5
    df['event_frequency_per_file'] = df.groupby('merge_key')['merge_key'].transform('count')
    
    # Calculate BOTH temporal windows (matching Phase 5 tuned model)
    print("   Calculating temporal windows (this may take a moment)...")
    df['events_in_1min_window'] = 0
    df['events_in_5min_window'] = 0
    
    # Convert to numpy for faster calculation
    case_times = pd.to_datetime(df['eventtime']).values
    
    for idx in range(len(df)):
        current_time = case_times[idx]
        if pd.notna(current_time):
            # 1-minute window (±30 seconds)
            window_1min_start = current_time - pd.Timedelta(seconds=30)
            window_1min_end = current_time + pd.Timedelta(seconds=30)
            window_1min_mask = (case_times >= window_1min_start) & (case_times <= window_1min_end)
            df.at[idx, 'events_in_1min_window'] = window_1min_mask.sum() - 1  # Exclude self
            
            # 5-minute window (±2.5 minutes)
            window_5min_start = current_time - pd.Timedelta(minutes=2.5)
            window_5min_end = current_time + pd.Timedelta(minutes=2.5)
            window_5min_mask = (case_times >= window_5min_start) & (case_times <= window_5min_end)
            df.at[idx, 'events_in_5min_window'] = window_5min_mask.sum() - 1  # Exclude self
    
    # Time since previous event
    time_diff = df['eventtime'].diff().dt.total_seconds()
    time_diff = time_diff.fillna(999999).astype('int64')
    df['time_since_previous_event_seconds'] = time_diff
    
    print(f"    ✓ Event frequency per file: max {df['event_frequency_per_file'].max():.0f}, mean {df['event_frequency_per_file'].mean():.1f}")
    print(f"    ✓ Events in 1-min window: max {df['events_in_1min_window'].max():.0f}")
    print(f"    ✓ Events in 5-min window: max {df['events_in_5min_window'].max():.0f}")
    
    return df

In [28]:
def extract_additional_features(df):
    """
    Extract remaining features for model compatibility.
    
    CRITICAL: Must match Phase 2A training dtypes EXACTLY.
    Some columns are INT64 (not bool) in training data!
    """
    print("\n  Extracting additional features...")
    
    # File system tunneling confidence (int64: 0 or 1)
    df['file_system_tunneling_confidence'] = 0  # int64, not float
    if 'is_tunneling' in df.columns:
        df.loc[df['is_tunneling'], 'file_system_tunneling_confidence'] = 1
    
    # CRITICAL FIX: Training data has INT64 (not bool) for these columns!
    df['has_attribute_change'] = df['lf_event'].fillna('').str.contains('Changing FileAttribute', case=False, na=False).astype('int64')
    df['has_timestamp_copied_from_file'] = df['copied_from_file'].fillna(False).astype('int64')
    df['zero_nanoseconds_logfile'] = df['zero_in_nanoseconds'].astype('int64')
    
    # Event vs modified time delta - FLOAT64
    df['event_vs_modified_after_days'] = None
    if 'lf_modified_time_after' in df.columns:
        modified_after = pd.to_datetime(df['lf_modified_time_after'], errors='coerce')
        event_time = pd.to_datetime(df['eventtime'], errors='coerce')
        df['event_vs_modified_after_days'] = (event_time - modified_after).dt.total_seconds() / 86400
    
    df['event_frequency_per_case'] = len(df)
    
    print(f"    ✓ Tunneling events: {df['file_system_tunneling_confidence'].sum():.0f}")
    print(f"    ✓ Attribute changes: {df['has_attribute_change'].sum():,} (dtype: {df['has_attribute_change'].dtype})")
    print(f"    ✓ Copied from file: {df['has_timestamp_copied_from_file'].sum():,} (dtype: {df['has_timestamp_copied_from_file'].dtype})")
    print(f"    ✓ Zero nanoseconds: {df['zero_nanoseconds_logfile'].sum():,} (dtype: {df['zero_nanoseconds_logfile'].dtype})")
    print(f"    ✓ Event frequency per case: {df['event_frequency_per_case'].iloc[0]:,}")
    
    return df

In [29]:
def extract_cross_artifact_features(df):
    """
    Extract cross-artifact validation scores using EXACT training logic.
    Phase 2A training uses this to calculate validation scores and pattern scores.
    """
    print("\n  Extracting cross-artifact features...")
    
    # UsnJrnl pattern flags
    df['usn_basic_info_change'] = df['usn_event_info'].fillna('').str.contains('Basic_Info_Change', case=False, na=False)
    df['usn_file_closed'] = df['usn_event_info'].fillna('').str.contains('File_Closed', case=False, na=False)
    df['usn_complete_manipulation_pattern'] = df['usn_basic_info_change'] & df['usn_file_closed']
    
    # Evidence indicators
    df['has_logfile_evidence'] = df['source'].isin(['both', 'logfile_only'])
    df['has_usnjrnl_evidence'] = df['source'].isin(['both', 'usnjrnl_only'])
    
    # Cross-artifact validation score
    def calculate_cross_artifact_score(row):
        if row['source'] == 'both':
            return 3
        elif row['source'] == 'logfile_only':
            return 2
        elif row['source'] == 'usnjrnl_only':
            if 'Basic_Info_Change' in str(row.get('usn_event_info', '')):
                return 1
        return 0
    
    df['cross_artifact_validation_score'] = df.apply(calculate_cross_artifact_score, axis=1)
    
    # Timestamp manipulation pattern score
    # NOTE: v2 uses events_in_5min_window (events_in_1min_window removed in Phase 3)
    def calculate_pattern_score(row):
        score = 0
        # Pattern 1: Rapid sequential manipulation (>10 events in 5min window)
        if row.get('events_in_5min_window', 0) > 10:
            score += 1
        # Pattern 2: Complete manipulation pattern
        if row.get('usn_complete_manipulation_pattern', False):
            score += 1
        # Pattern 3: Multiple manipulations on same file
        if row.get('event_frequency_per_file', 0) > 1:
            score += 1
        return min(score, 3)
    
    df['timestamp_manipulation_pattern_score'] = df.apply(calculate_pattern_score, axis=1)
    
    print(f"    ✓ Average cross-artifact score: {df['cross_artifact_validation_score'].mean():.2f}")
    print(f"    ✓ Average manipulation pattern score: {df['timestamp_manipulation_pattern_score'].mean():.2f}")
    
    return df

## 4. Apply Feature Engineering

In [30]:
print("=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)

# Convert eventtime to datetime if not already
df['eventtime'] = pd.to_datetime(df['eventtime'], errors='coerce')

# Apply all feature extraction functions
print("\n1. Parsing LogFile detail field...")
df = parse_lf_detail_field(df)

print("\n2. Extracting timestamp deltas...")
df = extract_timestamp_delta_features(df)

print("\n3. Extracting file characteristics...")
df = extract_file_characteristics(df)

print("\n4. Extracting temporal features...")
print("   (This may take a few minutes for large datasets...)")
df = extract_temporal_features(df)

print("\n5. Extracting cross-artifact features...")
df = extract_cross_artifact_features(df)

print("\n6. Extracting additional features...")
df = extract_additional_features(df)

print("\n" + "=" * 80)
print("✓ Feature engineering complete")
print("=" * 80)
print(f"\nTotal features: {len(df.columns)} columns")

FEATURE ENGINEERING

1. Parsing LogFile detail field...

  Parsing lf_detail field...
    ✓ Parsed 642 timestamp entries
    ✓ Zero nanoseconds: 309 events (dtype: bool)
    ✓ Copied timestamps: 262 events (dtype: bool)
    ✓ creation: 286 events (235 unique values)
    ✓ modified: 315 events (35 unique values)
    ✓ accessed: 14 events (5 unique values)
    ✓ mft_modified: 27 events (13 unique values)

2. Extracting timestamp deltas...

  Extracting timestamp delta features...
    ✓ Calculated 642 timestamp deltas

3. Extracting file characteristics...

  Extracting file characteristics...
    ✓ Suspicious extensions: 10,879 files
    ✓ Executable files: 0
    ✓ Average path depth: 6.5

4. Extracting temporal features...
   (This may take a few minutes for large datasets...)

  Extracting temporal features...
   Calculating temporal windows (this may take a moment)...
    ✓ Event frequency per file: max 2748, mean 582.9
    ✓ Events in 1-min window: max 5963
    ✓ Events in 5-min wind

## 5. Data Cleanup and Column Management

Remove unnecessary columns and add required columns for model compatibility.

In [31]:
print("=" * 80)
print("DATA CLEANUP")
print("=" * 80)

# Add missing columns
print("\n1. Adding missing columns...")
df['eventtime_dt'] = df['eventtime']
print("   ✓ Added eventtime_dt")

# Remove unnecessary columns
print("\n2. Removing columns not used in training...")

columns_to_remove = [
    'lf_detail', 'lf_creation_time', 'lf_modified_time', 'lf_accessed_time', 'lf_mft_modified_time',
    'usn_source_info', 'usn_carving_flag', 'usn_file_attribute',
    'lf_redo',
    'creation_time_delta_days', 'modified_time_delta_days',
    'time_until_next_event_seconds',
]

removed = []
for col in columns_to_remove:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)
        removed.append(col)

print(f"   ✓ Removed {len(removed)} columns:")
for col in removed:
    print(f"     - {col}")

# Rename columns
print("\n3. Renaming columns to match training...")

column_renames = {
    'usn_file_ref_num': 'usn_file_reference_number',
    'usn_parent_file_ref_num': 'usn_parent_file_reference_number'
}

renamed = []
for old_name, new_name in column_renames.items():
    if old_name in df.columns:
        df.rename(columns={old_name: new_name}, inplace=True)
        renamed.append(f"{old_name} → {new_name}")

if renamed:
    print(f"   ✓ Renamed {len(renamed)} columns:")
    for rename_info in renamed:
        print(f"     - {rename_info}")
else:
    print("   ✓ No columns needed renaming")

# Reorder columns to match training data
print("\n4. Reordering columns to match training data...")

column_order = [
    'eventtime', 'eventtime_dt', 'lf_lsn', 'lf_event', 'filename', 'filepath', 
    'lf_target_vcn', 'lf_cluster_index', 'merge_key', 'usn_usn', 'usn_event_info',
    'usn_file_reference_number', 'usn_parent_file_reference_number', 'source', 
    'is_tunneling', 'lf_creation_time_before', 'lf_creation_time_after',
    'lf_modified_time_before', 'lf_modified_time_after', 'lf_accessed_time_before',
    'lf_accessed_time_after', 'lf_mft_modified_time_before', 'lf_mft_modified_time_after',
    'zero_in_nanoseconds', 'copied_from_file', 'creation_time_changed_to_past',
    'modified_time_changed_to_past', 'accessed_time_delta_days', 'accessed_time_changed_to_past',
    'mft_modified_time_delta_days', 'mft_modified_time_changed_to_past', 'is_executable',
    'is_system_file', 'is_hidden_file', 'is_archive', 'filename_length',
    'has_suspicious_extension', 'event_frequency_per_file', 'event_frequency_per_case',
    'events_in_1min_window', 'events_in_5min_window', 'time_since_previous_event_seconds', 
    'has_logfile_evidence', 'has_usnjrnl_evidence', 'usn_basic_info_change', 'usn_file_closed',
    'usn_complete_manipulation_pattern', 'path_depth', 'event_vs_modified_after_days',
    'cross_artifact_validation_score', 'timestamp_manipulation_pattern_score',
    'file_system_tunneling_confidence', 'has_attribute_change', 
    'has_timestamp_copied_from_file', 'zero_nanoseconds_logfile'
]

existing_cols = [col for col in column_order if col in df.columns]
missing_cols = [col for col in column_order if col not in df.columns]

if missing_cols:
    print(f"   ⚠ WARNING: {len(missing_cols)} expected columns are missing:")
    for col in missing_cols[:5]:
        print(f"     - {col}")
    if len(missing_cols) > 5:
        print(f"     ... and {len(missing_cols) - 5} more")

df = df[existing_cols]
print(f"   ✓ Reordered {len(existing_cols)} columns to match training data")


print("\n" + "=" * 80)
print(f"✓ Cleanup complete: {len(df.columns)} columns remaining")
print("=" * 80)

DATA CLEANUP

1. Adding missing columns...
   ✓ Added eventtime_dt

2. Removing columns not used in training...
   ✓ Removed 11 columns:
     - lf_detail
     - lf_creation_time
     - lf_modified_time
     - lf_accessed_time
     - lf_mft_modified_time
     - usn_source_info
     - usn_carving_flag
     - usn_file_attribute
     - lf_redo
     - creation_time_delta_days
     - modified_time_delta_days

3. Renaming columns to match training...
   ✓ Renamed 2 columns:
     - usn_file_ref_num → usn_file_reference_number
     - usn_parent_file_ref_num → usn_parent_file_reference_number

4. Reordering columns to match training data...
   ✓ Reordered 55 columns to match training data

✓ Cleanup complete: 55 columns remaining


## 6. Save Features

In [32]:
print("=" * 80)
print("SAVING FEATURES")
print("=" * 80)

output_file = OUTPUT_DIR / 'forensic_features.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n✓ Saved features to:")
print(f"   {output_file}")
print(f"\n   Total records: {len(df):,}")
print(f"   Total columns: {len(df.columns)}")
print(f"\n✓ Ready for detection (Step 3)")

SAVING FEATURES

✓ Saved features to:
   /Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/Lone-Wolf/forensic_features.csv

   Total records: 33,888
   Total columns: 55

✓ Ready for detection (Step 3)


## 7. Feature Summary

In [33]:
print("=" * 80)
print("FEATURE SUMMARY")
print("=" * 80)

# List all engineered features
base_cols = ['lf_lsn', 'usn_usn', 'eventtime', 'filename', 'filepath', 'lf_event', 'lf_detail', 
             'usn_event_info', 'usn_source_info', 'usn_file_attribute', 'merge_key', 'source']
feature_cols = [col for col in df.columns if col not in base_cols]

print(f"\nEngineered Features ({len(feature_cols)} total):\n")
for i, col in enumerate(sorted(feature_cols), 1):
    non_null = df[col].notna().sum()
    pct = non_null / len(df) * 100
    print(f"{i:2}. {col:45} {non_null:6,} / {len(df):,} ({pct:5.1f}%)")

print("\n" + "=" * 80)
print("✓ Feature engineering complete!")
print("=" * 80)

FEATURE SUMMARY

Engineered Features (46 total):

 1. accessed_time_changed_to_past                 33,888 / 33,888 (100.0%)
 2. accessed_time_delta_days                          14 / 33,888 (  0.0%)
 3. copied_from_file                              33,888 / 33,888 (100.0%)
 4. creation_time_changed_to_past                 33,888 / 33,888 (100.0%)
 5. cross_artifact_validation_score               33,888 / 33,888 (100.0%)
 6. event_frequency_per_case                      33,888 / 33,888 (100.0%)
 7. event_frequency_per_file                      33,888 / 33,888 (100.0%)
 8. event_vs_modified_after_days                     180 / 33,888 (  0.5%)
 9. events_in_1min_window                         33,888 / 33,888 (100.0%)
10. events_in_5min_window                         33,888 / 33,888 (100.0%)
11. eventtime_dt                                  33,753 / 33,888 ( 99.6%)
12. file_system_tunneling_confidence              33,888 / 33,888 (100.0%)
13. filename_length                               

---

## ✓ Step 2 Complete!

**Next:** Run `03_Run_Detection.ipynb` to apply the trained model and detect timestomped files.

---